# Fama–French factor model — 50-name real panel

**On the day: this is notebook `01`.** Then Gu (`01a`). Home-ground FF, then lagged ML, then the paper.

**Papers (original titles):**

- Fama, E. F., & French, K. R. (1993). *Common risk factors in the returns on stocks and bonds.* *Journal of Financial Economics*, 33(1), 3–56.
- Fama, E. F., & French, K. R. (2015). *A five-factor asset pricing model.* *Journal of Financial Economics*, 116(1), 1–22.

Classic **time-series** Fama–French: the Ken French factors are **one path for the whole market**. Each stock gets its own \(\alpha\) and \(\beta\).

This is **not** notebook `01b` (synthetic *characteristics*). No Compustat.

**Data already in the repo**

| File | Role |
|------|------|
| `data/krauss_sp500_daily.csv` | Yahoo adjusted close, same 50 names + SPY as Krauss |
| `data/ff_daily_factors.csv` | Daily `mkt_rf`, `smb`, `hml`, `rmw`, `cma`, `rf`, `mom` (decimals) |

SPY is dropped from the stock table (French already supplies the market factor).

**Two tasks in one notebook**

1. Contemporaneous FF OLS — *explain* today’s excess return (no ML).  
2. Ridge / RF on **yesterday’s** features — *forecast* today’s excess (this is the ML).

**Companion:** `sessions/01_ff_factors_real.md` · `slides/01_ff_factors.md`


## 0. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from scipy.stats import f as f_dist
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True


def find_csv(rel: str) -> Path:
    rel = Path(rel)
    for base in [Path.cwd(), *Path.cwd().parents]:
        p = base / rel
        if p.is_file():
            return p.resolve()
    raise FileNotFoundError(f"Missing {rel} (cwd={Path.cwd()})")


def display_path(p):
    p = Path(p).resolve()
    for base in p.parents:
        if (base / "requirements.txt").is_file():
            try:
                return p.relative_to(base).as_posix()
            except ValueError:
                break
    return p.name


try:
    PRICES = find_csv("data/krauss_sp500_daily.csv")
except FileNotFoundError:
    raise FileNotFoundError(
        "Price file not found. Run: python data/download_krauss_panel.py, then rerun this notebook."
    ) from None
FACTORS = find_csv("data/ff_daily_factors.csv")
print("prices ", display_path(PRICES))
print("factors", display_path(FACTORS))


## 1. Excess returns + merge factors

\[
r_{i,t}-r_{f,t}
= \alpha_i + \beta_i'\mathrm{FF}_t + \varepsilon_{i,t}
\]

Factors are contemporaneous — this **explains** today’s return, it does not forecast \(r_{t+1}\) the way Gu does.


In [ ]:
prices = pd.read_csv(PRICES, parse_dates=["date"])
ff = pd.read_csv(FACTORS, parse_dates=["date"])

px = prices.loc[prices["ticker"] != "SPY"].sort_values(["ticker", "date"])
px["ret"] = px.groupby("ticker")["close"].pct_change()
panel = px.dropna(subset=["ret"]).merge(ff, on="date", how="inner")
panel["excess"] = panel["ret"] - panel["rf"]

print(panel[["ticker", "date"]].nunique().to_dict())
print(panel["date"].min().date(), "→", panel["date"].max().date())
print("rows", len(panel))
panel.head()

## 2. Calendar split

Estimate betas **only** on 2015–2019. 2020–2023 is the holdout (COVID + aftermath). Same idea as Gu: no shuffled days.


In [ ]:
train_end = pd.Timestamp("2019-12-31")
train = panel[panel["date"] <= train_end]
test = panel[panel["date"] > train_end]
print(f"train {train['date'].min().date()} → {train['date'].max().date()}  rows={len(train):,}")
print(f"test  {test['date'].min().date()} → {test['date'].max().date()}  rows={len(test):,}")

FF3 = ["mkt_rf", "smb", "hml"]
FF5 = FF3 + ["rmw", "cma"]

## 3. OLS helper (intercept = \(\alpha\))

In [ ]:
def ols(y, X):
    """y = a + X b + e. Returns dict with alpha, betas, t-stats, R2."""
    y = np.asarray(y, float)
    X = np.asarray(X, float)
    Z = np.column_stack([np.ones(len(y)), X])
    n, k = Z.shape
    beta, *_ = np.linalg.lstsq(Z, y, rcond=None)
    resid = y - Z @ beta
    dof = max(n - k, 1)
    s2 = float(resid @ resid) / dof
    xtx_inv = np.linalg.inv(Z.T @ Z)
    se = np.sqrt(np.maximum(s2 * np.diag(xtx_inv), 0))
    tstat = beta / np.where(se > 0, se, np.nan)
    sst = float((y - y.mean()) @ (y - y.mean()))
    r2 = 1.0 - float(resid @ resid) / sst if sst > 0 else np.nan
    return {"coef": beta, "se": se, "tstat": tstat, "r2": r2, "n": n}


def r2_oos(y, yhat):
    y, yhat = np.asarray(y, float), np.asarray(yhat, float)
    sst = float((y - y.mean()) @ (y - y.mean()))
    return 1.0 - float((y - yhat) @ (y - yhat)) / sst if sst > 0 else np.nan


def grs_test(df, cols):
    """Gibbons–Ross–Shanken (1989): H0 that all intercepts are zero.
    Balanced panel (inner-join dates). Returns GRS ~ F(N, T-N-K), p-value, mean |α|.
    """
    y_wide = df.pivot(index="date", columns="ticker", values="excess")
    f_wide = df.drop_duplicates("date").set_index("date")[list(cols)].sort_index()
    data = y_wide.join(f_wide, how="inner").dropna(how="any")
    tickers = [c for c in y_wide.columns if c in data.columns]
    Y = data[tickers].to_numpy()
    Fmat = data[list(cols)].to_numpy()
    T, N = Y.shape
    K = Fmat.shape[1]
    Z = np.column_stack([np.ones(T), Fmat])
    B, *_ = np.linalg.lstsq(Z, Y, rcond=None)
    E = Y - Z @ B
    alpha = B[0]
    # MLE covariances: divide by T (FF / GRS convention in the session note)
    Sigma = (E.T @ E) / T
    mu_f = Fmat.mean(axis=0)
    Fc = Fmat - mu_f
    Omega = (Fc.T @ Fc) / T
    grs = ((T - N - K) / N) * float(alpha @ np.linalg.solve(Sigma, alpha)) / (
        1.0 + float(mu_f @ np.linalg.solve(Omega, mu_f))
    )
    pval = float(1.0 - f_dist.cdf(grs, N, T - N - K))
    sst = ((Y - Y.mean(axis=0)) ** 2).sum(axis=0)
    ssr = (E ** 2).sum(axis=0)
    mean_r2 = float(np.nanmean(np.where(sst > 0, 1.0 - ssr / sst, np.nan)))
    mean_abs_a = float(np.mean(np.abs(alpha)))
    return {
        "T": T, "N": N, "K": K,
        "GRS": float(grs),
        "p": pval,
        "mean_abs_alpha": mean_abs_a,
        "mean_abs_alpha_ann": mean_abs_a * 252.0,
        "mean_r2": mean_r2,
    }

## 4. One name in detail (AAPL, FF3)

Read the intercept as **training-sample alpha** (daily), not a tradable forecast.


In [ ]:
def window(df, ticker, split):
    return df.loc[df["ticker"] == ticker]


aapl_tr = window(train, "AAPL", "train")
fit_aapl = ols(aapl_tr["excess"], aapl_tr[FF3])
names = ["alpha"] + FF3
print(pd.DataFrame({
    "coef": fit_aapl["coef"],
    "tstat": fit_aapl["tstat"],
}, index=names).round(4))
print(f"train R2 = {fit_aapl['r2']:.3f}   n = {fit_aapl['n']}")

## 5. All names — FF3 and FF5

Per ticker: fit on train, freeze \(\beta\), score explanatory \(R^2\) on test.

FF (2015) do **not** pick a winner with \(R^2\). They look at intercepts: the **GRS** test that all \(\alpha_i=0\), and mean \(|\alpha|\). We report both next to \(R^2\).


**Survivorship.** These 50 names are today’s large firms from Yahoo — they survived and grew. Their \(\hat\alpha_i\) are biased **upward**. That is a sample fact, not evidence that FF3 or FF5 is mis-specified. The same bias comes back in real-mode Krauss (`02`).


In [ ]:
def fit_ticker(tr, te, cols):
    if len(tr) < 60 or len(te) < 40:
        return None
    fit = ols(tr["excess"], tr[cols])
    yhat = te[cols].to_numpy() @ fit["coef"][1:]  # freeze train beta; leave alpha out of the forecast
    alpha_test = float(np.mean(te["excess"].to_numpy() - yhat))
    return {
        "alpha_train": fit["coef"][0],
        "t_alpha": fit["tstat"][0],
        "alpha_test": alpha_test,
        "r2_train": fit["r2"],
        "r2_test": r2_oos(te["excess"], yhat + fit["coef"][0]),
        "n_train": fit["n"],
        "n_test": len(te),
    }


rows = []
for ticker, g in panel.groupby("ticker"):
    tr, te = g[g["date"] <= train_end], g[g["date"] > train_end]
    s3 = fit_ticker(tr, te, FF3)
    s5 = fit_ticker(tr, te, FF5)
    if s3 is None or s5 is None:
        continue
    rows.append({
        "ticker": ticker,
        "alpha_train_ff3": s3["alpha_train"],
        "alpha_train_ff5": s5["alpha_train"],
        "alpha_test_ff3": s3["alpha_test"],
        "alpha_test_ff5": s5["alpha_test"],
        "t_alpha": s3["t_alpha"],
        "r2_train_ff3": s3["r2_train"],
        "r2_test_ff3": s3["r2_test"],
        "r2_train_ff5": s5["r2_train"],
        "r2_test_ff5": s5["r2_test"],
    })

board = pd.DataFrame(rows).sort_values("r2_train_ff3", ascending=False)
print(board.round(4).head(10).to_string(index=False))
print("...")
print("Mean R²")
print(board[["r2_train_ff3", "r2_test_ff3", "r2_train_ff5", "r2_test_ff5"]].mean().round(3).to_string())
print("Mean |alpha| annualized (×252)")
print((board[["alpha_train_ff3", "alpha_test_ff3", "alpha_train_ff5", "alpha_test_ff5"]].abs().mean() * 252).round(4).to_string())

# GRS + mean |α| + mean R² by model and period (re-estimated in each window; MLE cov)
sum_rows = []
for lab, cols in [("FF3", FF3), ("FF5", FF5)]:
    for period, dfp in [("train", train), ("test", test)]:
        g = grs_test(dfp, cols)
        sum_rows.append({
            "model": lab,
            "period": period,
            "GRS": g["GRS"],
            "p_value": g["p"],
            "mean_|alpha|_ann": g["mean_abs_alpha_ann"],
            "mean_R2": g["mean_r2"],
            "T": g["T"], "N": g["N"], "L": g["K"],
        })
summary = pd.DataFrame(sum_rows)
print("\nmodel × period (GRS uses MLE covariances, divide by T)")
print(summary.round(4).to_string(index=False))
print("n tickers", len(board))
summary.round(4)

## 6. Pictures

In [ ]:
fig, ax = plt.subplots()
ax.hist(board["alpha_train_ff3"] * 252, bins=15, edgecolor="white")
ax.set_xlabel("train alpha (annualised, ×252, naive)")
ax.set_title("Distribution of FF3 training alphas")
plt.tight_layout()

fig, ax = plt.subplots()
ax.scatter(board["r2_train_ff3"], board["r2_test_ff3"], alpha=0.75)
ax.plot([0, 1], [0, 1], ls="--", c="0.5", lw=1)
ax.set_xlabel("train $R^2$ (FF3)")
ax.set_ylabel("test R2 (FF3, frozen beta)")
ax.set_title("Does in-sample fit survive 2020–2023?")
plt.tight_layout()

## 7. Discussion questions

1. Why is `mkt_rf` the same number for AAPL and for JPM on a given day?
2. What would a large **train** \(\alpha\) mean — and why is that not Gu’s \(\hat r_{t+1}\)?
3. FF5 vs FF3: who wins on **GRS / mean \(|\alpha|\)**? Who wins on test \(R^2\)? Why are those different questions?
4. After §8: why is lagged Ridge/RF \(R^2\) tiny next to contemporaneous FF \(R^2\)? Which one is a forecast?


## 8. Machine learning — lagged features (forecast, not attribution)

Contemporaneous FF uses **today’s** factors to explain today’s return. That is not a trading signal.

Now: features known at \(t-1\), target = excess return at \(t\). Same calendar cut.

| Feature | Why |
|---------|-----|
| Lagged FF (`mkt_rf`, `smb`, `hml`, `rmw`, `cma`, `mom`) | Market-wide; same \(X\) for every name that day |
| Own `ret` lag-1 and 5-day mean | Stock-specific (otherwise Ridge/RF cannot rank names) |

Pooled panel (one model, all stocks) — closer to Gu than 50 separate regressions.

(We discuss this result in class.)


In [ ]:
lag_ff = ["mkt_rf", "smb", "hml", "rmw", "cma", "mom"]
ml = panel.sort_values(["ticker", "date"]).copy()
for c in lag_ff + ["ret"]:
    ml[f"{c}_l1"] = ml.groupby("ticker")[c].shift(1)
ml["ret_l5"] = ml.groupby("ticker")["ret"].transform(
    lambda s: s.shift(1).rolling(5, min_periods=5).mean()
)

feat = [f"{c}_l1" for c in lag_ff] + ["ret_l1", "ret_l5"]
ml = ml.dropna(subset=feat + ["excess"])

ml_tr = ml[ml["date"] <= train_end]
ml_te = ml[ml["date"] > train_end]
X_tr, y_tr = ml_tr[feat], ml_tr["excess"].to_numpy()
X_te, y_te = ml_te[feat], ml_te["excess"].to_numpy()
print("features:", feat)
print(f"train rows={len(ml_tr):,}  test rows={len(ml_te):,}")


def r2_vs0(y, yhat):
    y, yhat = np.asarray(y), np.asarray(yhat)
    return 1.0 - np.sum((y - yhat) ** 2) / np.sum(y ** 2)


candidates = {
    "OLS_lag": LinearRegression(),
    "Ridge": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=10.0)),
    ]),
    "RF": RandomForestRegressor(
        n_estimators=150, max_depth=4, min_samples_leaf=50,
        random_state=2020, n_jobs=-1,
    ),
}

rows, preds_te = [], {}
for name, model in candidates.items():
    model.fit(X_tr, y_tr)
    p_tr, p_te = model.predict(X_tr), model.predict(X_te)
    preds_te[name] = p_te
    rows.append({
        "model": name,
        "r2_vs0_train": r2_vs0(y_tr, p_tr),
        "r2_vs0_test": r2_vs0(y_te, p_te),
    })

ml_board = pd.DataFrame(rows).sort_values("r2_vs0_test", ascending=False)
print(ml_board.round(4).to_string(index=False))
print("\nCompare: mean contemporaneous FF3 *explanatory* R2 on test was ~0.48.")
print("Here R2 is Gu-style predictive (vs a 0 forecast).")

# Daily long-short on the best test model (equal weight, 10% tails)
best = ml_board.iloc[0]["model"]
tmp = ml_te[["date", "ticker", "ret"]].copy()
tmp["pred"] = preds_te[best]
ls = []
for dt, g in tmp.groupby("date"):
    if len(g) < 10:
        continue
    lo, hi = g["pred"].quantile(0.1), g["pred"].quantile(0.9)
    long = g.loc[g["pred"] >= hi, "ret"].mean()
    short = g.loc[g["pred"] <= lo, "ret"].mean()
    ls.append(long - short)
ls = np.asarray(ls)
print(f"\nLong-short on {best} (q=0.1), test days={len(ls)}")
print("mean daily LS:", round(ls.mean(), 5), "  ann. Sharpe (naive):",
      round(np.sqrt(252) * ls.mean() / ls.std(ddof=1), 3) if ls.std() > 0 else np.nan)

## Takeaway

**FF OLS** (earlier cells) attributes today’s return to today’s factors — high explanatory \(R^2\), no ML.

**Ridge / RF** on yesterday’s factors + own lags is the ML piece: same panel, same split, but a **forecast**. (We discuss this result in class.)

Gu’s next move: a larger *characteristic* zoo and a monthly horizon — not a fancier fit of today’s SMB.

**Siblings:** `01a_gu_kelly_xiu_simplified.ipynb` · `01b_gu_fama_french.ipynb`
